# Medical Image Diagnosis — System Demo

This notebook demonstrates the full pipeline **without needing trained weights**:
- Augmentation pipeline with sample images
- Model architecture (ViT-B/16 and EfficientNet-B3)
- Loss functions and metrics
- Grad-CAM visualization
- Inference predictor

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import yaml

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

## 1. Configuration

In [ ]:
with open('../configs/chest_xray.yaml') as f:
    cfg = yaml.safe_load(f)

print('Dataset:', cfg.get('project', {}).get('name'))
print('Classes:', cfg.get('dataset', {}).get('classes'))
print('Model:  ', cfg.get('model', {}).get('name'))
print('Image size:', cfg.get('model', {}).get('image_size'))

## 2. Augmentation Pipeline

In [ ]:
from preprocessing.augmentation import build_augmentation_pipeline
from preprocessing.transforms import denormalize

# Simulate a chest X-ray (grayscale → RGB)
np.random.seed(42)
fake_xray = np.random.randint(30, 200, (512, 512), dtype=np.uint8)
fake_xray = np.stack([fake_xray] * 3, axis=-1)  # grayscale → RGB

train_transform = build_augmentation_pipeline(cfg, split='train')
eval_transform = build_augmentation_pipeline(cfg, split='val')

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(fake_xray, cmap='gray')
axes[0].set_title('Original', fontsize=12)
axes[0].axis('off')

for i in range(1, 5):
    aug = train_transform(image=fake_xray)
    vis = denormalize(aug['image'])
    axes[i].imshow(vis, cmap='gray')
    axes[i].set_title(f'Augmented #{i}', fontsize=12)
    axes[i].axis('off')

plt.suptitle('Training Augmentation Pipeline (Albumentations)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 3. Model Architecture

In [ ]:
from models.vit import build_vit
from models.efficientnet import build_efficientnet

# Build both architectures (pretrained=False for speed)
cfg_demo = dict(cfg)
cfg_demo['model'] = {**cfg.get('model', {}), 'pretrained': False, 'name': 'vit_b16', 'image_size': 224}

vit = build_vit(cfg_demo, num_classes=2)
print(f'ViT-B/16  — Total params: {vit.count_params()/1e6:.1f}M')

cfg_demo['model']['name'] = 'efficientnet_b0'
eff = build_efficientnet(cfg_demo, num_classes=2)
print(f'EfficientNet-B0 — Total params: {eff.count_params()/1e6:.1f}M')

# Forward pass
dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    vit_out = vit(dummy)
    eff_out = eff(dummy)
print(f'\nViT output shape:  {vit_out.shape}  (batch=2, classes=2)')
print(f'EffNet output shape: {eff_out.shape}  (batch=2, classes=2)')

## 4. Loss Functions

In [ ]:
from models.losses import FocalLoss, LabelSmoothingCrossEntropy, build_loss
import torch.nn as nn

logits = torch.randn(8, 2)
labels = torch.randint(0, 2, (8,))

losses = {
    'Cross-Entropy': nn.CrossEntropyLoss()(logits, labels),
    'Focal Loss (γ=2)': FocalLoss(gamma=2.0)(logits, labels),
    'Label Smoothing (0.1)': LabelSmoothingCrossEntropy(smoothing=0.1)(logits, labels),
}

print('Loss function comparison on random batch:')
for name, val in losses.items():
    print(f'  {name:30s}: {val.item():.4f}')

## 5. Evaluation Metrics

In [ ]:
from models.metrics import MetricsCalculator

# Simulate a near-perfect classifier
np.random.seed(0)
n = 100
true_labels = torch.randint(0, 2, (n,))
# 90% correct predictions
pred_logits = torch.zeros(n, 2)
for i, t in enumerate(true_labels):
    if np.random.rand() < 0.9:
        pred_logits[i, t] = 5.0   # correct
    else:
        pred_logits[i, 1 - t] = 5.0  # wrong

calc = MetricsCalculator(num_classes=2, class_names=['Normal', 'Pneumonia'])
calc.update(pred_logits, true_labels)
metrics = calc.compute()

print('Metrics (simulated 90% accurate classifier):')
for k, v in metrics.items():
    print(f'  {k:20s}: {v:.4f}')

## 6. Confusion Matrix & Class Report

In [ ]:
import seaborn as sns

cm = calc.confusion_matrix()
print(calc.classification_report())

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Pneumonia'],
            yticklabels=['Normal', 'Pneumonia'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

## 7. Grad-CAM on a Random Tensor (EfficientNet)

In [ ]:
from explainability.gradcam import GradCAMPlusPlus, get_target_layer
import cv2

cfg_demo['model']['name'] = 'efficientnet_b0'
model = build_efficientnet(cfg_demo, num_classes=2)
model.eval()

target_layer = get_target_layer(model)
gradcam = GradCAMPlusPlus(model, target_layer)

# Use a random tensor as if it were a real image
input_tensor = torch.randn(1, 3, 224, 224)
cam, pred_class, probs = gradcam.generate(input_tensor)

# Create a fake X-ray-like background image for visualization
fake_img = np.random.randint(50, 180, (224, 224, 3), dtype=np.uint8)
overlay = gradcam.overlay_on_image(fake_img, cam)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(fake_img, cmap='gray')
axes[0].set_title('Input Image')
axes[1].imshow(cam, cmap='jet')
axes[1].set_title('Grad-CAM++ Heatmap')
axes[2].imshow(overlay)
axes[2].set_title(f'Overlay — Predicted: class {pred_class}\n{probs}')
for ax in axes: ax.axis('off')
plt.suptitle('Grad-CAM++ Explainability (EfficientNet-B0)', fontsize=13)
plt.tight_layout()
plt.show()

gradcam.remove_hooks()

## 8. Project Summary

In [ ]:
import subprocess, os
result = subprocess.run(['git', 'log', '--oneline'], capture_output=True, text=True, cwd='..')
print('Git commit history:')
print(result.stdout)

print('\nProject file structure:')
for root, dirs, files in os.walk('..'):
    dirs[:] = [d for d in dirs if d not in ('__pycache__', '.git', '.mypy_cache', 'checkpoints', 'datasets', 'results', 'wandb', 'logs')]
    level = root.replace('..', '').count(os.sep)
    indent = '  ' * level
    folder = os.path.basename(root)
    if level <= 2:
        print(f'{indent}{folder}/')
    for f in sorted(files):
        if not f.endswith(('.pyc',)):
            print(f'{indent}  {f}')